# Projet du module 3 — Classer et regrouper les chiffres de MNIST

Carnet de départ de l'équipe. Remplacez chaque « À vous » par votre code, gardez les sorties, et rédigez vos analyses dans le rapport de la page du projet.

**Avant de commencer** : *Exécution → Modifier le type d'exécution → GPU (T4)*, puis *Fichier → Enregistrer une copie dans Drive* (une copie partagée par l'équipe).

Équipe : (vos noms)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, adjusted_rand_score, normalized_mutual_info_score
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
keras.utils.set_random_seed(42)

## Fonctions fournies : exporter le modèle pour l'interface

Exécutez cette cellule telle quelle. `exporter` (Keras) ou `exporter_pytorch` écrit `modele_mnist.json`; `verifier_export` refait le calcul de l'interface en NumPy et le compare à votre modèle.

In [ ]:
import json
import numpy as np

def _liste(a, decimales=5):
    return np.round(np.asarray(a, dtype=float), decimales).tolist()

def exporter(modele, fichier="modele_mnist.json", couche_caracteristiques="caracteristiques", kmeans=None,
             etiquettes_groupes=None, equipe=""):
    """Exporte un réseau Keras (Conv2D, MaxPooling2D, Flatten, Dense, Dropout) et des k-moyennes, pour l'interface du site."""
    couches, indice = [], None
    for couche in modele.layers:
        nom, config = couche.__class__.__name__, couche.get_config()
        if nom in ("Dropout", "InputLayer", "RandomRotation", "RandomTranslation", "RandomZoom"):
            pass                                                   # sans effet à la prédiction
        elif nom == "Reshape" and tuple(config["target_shape"]) == (28, 28, 1):
            pass
        elif nom == "Conv2D":
            if tuple(config["strides"]) != (1, 1):
                raise ValueError("l'interface accepte seulement des convolutions de pas 1")
            k, b = couche.get_weights()
            couches.append({"type": "conv2d", "forme": list(k.shape), "noyaux": _liste(k), "biais": _liste(b),
                            "padding": config["padding"], "activation": config["activation"]})
        elif nom == "MaxPooling2D":
            couches.append({"type": "maxpool", "taille": int(config["pool_size"][0])})
        elif nom == "Flatten":
            couches.append({"type": "flatten", "ordre": "hwc"})
        elif nom == "Dense":
            w, b = couche.get_weights()
            couches.append({"type": "dense", "forme": list(w.shape), "poids": _liste(w), "biais": _liste(b),
                            "activation": config["activation"]})
        else:
            raise ValueError(f"couche {nom} non prise en charge par l'interface (Conv2D, MaxPooling2D, Flatten, Dense, Dropout)")
        if couche.name == couche_caracteristiques:
            indice = len(couches) - 1
    _ecrire(fichier, couches, indice, kmeans, etiquettes_groupes, equipe)

def exporter_pytorch(reseau, fichier="modele_mnist.json", indice_caracteristiques=None, kmeans=None,
                     etiquettes_groupes=None, equipe=""):
    """Même chose pour un nn.Sequential PyTorch (Conv2d, ReLU, MaxPool2d, Flatten, Linear, Dropout, Softmax).
    indice_caracteristiques : la position, dans le Sequential, du module qui produit les caractéristiques."""
    import torch.nn as nn
    couches, indice = [], None
    for i, m in enumerate(reseau):
        if isinstance(m, nn.Conv2d):
            k = m.weight.detach().cpu().numpy().transpose(2, 3, 1, 0)        # (h, w, entrée, sortie)
            p = m.padding if isinstance(m.padding, str) else ("same" if m.padding[0] == (k.shape[0] - 1) // 2 > 0 else "valid")
            couches.append({"type": "conv2d", "forme": list(k.shape), "noyaux": _liste(k),
                            "biais": _liste(m.bias.detach().cpu().numpy()), "padding": p, "activation": "linear"})
        elif isinstance(m, (nn.ReLU, nn.Softmax, nn.Sigmoid, nn.Tanh)):
            couches[-1]["activation"] = {nn.ReLU: "relu", nn.Softmax: "softmax", nn.Sigmoid: "sigmoid", nn.Tanh: "tanh"}[type(m)]
        elif isinstance(m, nn.MaxPool2d):
            couches.append({"type": "maxpool", "taille": int(m.kernel_size if isinstance(m.kernel_size, int) else m.kernel_size[0])})
        elif isinstance(m, nn.Flatten):
            couches.append({"type": "flatten", "ordre": "chw"})
        elif isinstance(m, nn.Linear):
            w = m.weight.detach().cpu().numpy().T
            couches.append({"type": "dense", "forme": list(w.shape), "poids": _liste(w),
                            "biais": _liste(m.bias.detach().cpu().numpy()), "activation": "linear"})
        elif isinstance(m, nn.Dropout):
            pass
        else:
            raise ValueError(f"module {type(m).__name__} non pris en charge par l'interface")
        if i == indice_caracteristiques:
            indice = len(couches) - 1
    _ecrire(fichier, couches, indice, kmeans, etiquettes_groupes, equipe)

def _ecrire(fichier, couches, indice, kmeans, etiquettes_groupes, equipe):
    sortie = {"equipe": str(equipe), "entree": [28, 28, 1], "couches": couches}
    if kmeans is not None:
        if indice is None:
            raise ValueError("couche des caractéristiques introuvable : vérifiez son nom (ou son indice)")
        sortie["caracteristiques"] = indice
        sortie["centres"] = _liste(kmeans.cluster_centers_)
        if etiquettes_groupes is not None:
            sortie["etiquettes_groupes"] = [int(e) for e in etiquettes_groupes]
    with open(fichier, "w") as f:
        json.dump(sortie, f)
    import os
    print(f"{fichier} : {len(couches)} couches, {os.path.getsize(fichier) / 1e6:.1f} Mo")

def predire_export(fichier, images):
    """Refait en NumPy le calcul de l'interface du site, à partir du fichier exporté (images : n × 28 × 28 × 1)."""
    m = json.load(open(fichier))
    act = {"relu": lambda z: np.maximum(z, 0), "linear": lambda z: z, "sigmoid": lambda z: 1 / (1 + np.exp(-z)),
           "tanh": np.tanh, "softmax": lambda z: np.exp(z - z.max(-1, keepdims=True)) / np.exp(z - z.max(-1, keepdims=True)).sum(-1, keepdims=True)}
    t = np.asarray(images, dtype=float).reshape(len(images), 28, 28, 1)
    for c in m["couches"]:
        if c["type"] == "conv2d":
            k, b = np.array(c["noyaux"]), np.array(c["biais"])
            kh, kw = k.shape[:2]
            if c["padding"] == "same":
                t = np.pad(t, ((0, 0), ((kh - 1) // 2, kh // 2), ((kw - 1) // 2, kw // 2), (0, 0)))
            H, W = t.shape[1] - kh + 1, t.shape[2] - kw + 1
            s = sum(np.tensordot(t[:, u:u + H, v:v + W, :], k[u, v], axes=([3], [0])) for u in range(kh) for v in range(kw))
            t = act[c["activation"]](s + b)
        elif c["type"] == "maxpool":
            p = c["taille"]; n, H, W, C = t.shape
            t = t[:, :H // p * p, :W // p * p].reshape(n, H // p, p, W // p, p, C).max(axis=(2, 4))
        elif c["type"] == "flatten":
            t = t.transpose(0, 3, 1, 2).reshape(len(t), -1) if c["ordre"] == "chw" else t.reshape(len(t), -1)
        else:
            t = act[c["activation"]](t @ np.array(c["poids"]) + np.array(c["biais"]))
    return t

def verifier_export(fichier, modele_predict, images):
    """Compare les prédictions du fichier exporté à celles du modèle (images : n × 28 × 28 × 1)."""
    a = predire_export(fichier, images).argmax(1)
    b = np.asarray(modele_predict(images)).argmax(1)
    print(f"mêmes prédictions pour {np.mean(a == b) * 100:.1f} % des {len(images)} images vérifiées")
    return bool(np.all(a == b))

## Partie 1 — Préparer les données

In [ ]:
# À vous : charger MNIST, normaliser, forme 28 × 28 × 1, validation stratifiée, exemples et répartition des classes

## Partie 2 — Classer les chiffres

D'abord un perceptron multicouche (la référence), puis un réseau convolutif. Nommez `caracteristiques` la couche dense juste avant la sortie.

In [ ]:
# À vous : le perceptron multicouche de référence

In [ ]:
# À vous : le réseau convolutif, ses courbes d'apprentissage, la comparaison sur la validation

In [ ]:
# À vous : mesure finale sur le test (une seule fois), matrice de confusion, analyse des erreurs

## Partie 3 — Le regroupement profond

In [ ]:
# À vous : caractéristiques de la couche « caracteristiques », k-moyennes (k = 10) sur les caractéristiques et sur les pixels (ACP 50),
# ARI et NMI, visualisation t-SNE, puis un k plus grand et les images proches des centres

## Partie 4 — Visualiser pour convaincre

In [ ]:
# À vous : tableau croisé groupe × vrai chiffre (carte de chaleur), tailles des groupes, et vos autres figures

## Partie 5 — Exporter le modèle pour l'interface

Exportez le réseau et les k-moyennes (k = 10), vérifiez l'exportation, puis téléchargez `modele_mnist.json` (panneau *Fichiers* à gauche) et chargez-le dans l'interface de la page du projet.

In [ ]:
# À vous : exporter(cnn, kmeans=km, etiquettes_groupes=..., equipe=...), puis verifier_export(...)